# Rent Estimator in Colab
Run the cells in order. Step 3 prints a link to your app.

Addresses are loaded with pandas from the SF open-data CSV. Every other value is a placeholder.

## Step 1: Install
Colab already has pandas; this adds Streamlit.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
!pip install -q streamlit

In [ ]:
import pandas as pd
df = pd.read_csv("https://data.sf.gov/api/v3/views/ramy-di5m/export.csv?accessType=DOWNLOAD")

In [ ]:
df_web = df[['Address', 'Latitude', 'Longitude', 'nhood']]

In [ ]:
df_web.to_csv("df_web.csv", index=False)

## Step 2: Write the app file

In [ ]:
%%writefile app.py
import pandas as pd
import streamlit as st
import random

st.set_page_config(page_title="Rent Estimator", layout="wide")

# ------------------------------------------------------------------
# Load data
# ------------------------------------------------------------------

@st.cache_data
def load_data():
    return pd.read_csv("df_web.csv")

df_web = load_data()

# ------------------------------------------------------------------
# Address search
# ------------------------------------------------------------------

st.title("What should this listing rent for?")

st.caption("Enter an address to view property information and estimate rent.")

text = st.text_input("Address", placeholder="Start typing an address...")

if not text:
    st.info("Type part of an address to search.")
    st.stop()

matches = df_web[df_web["Address"].str.contains(text, case=False, regex=False)].head(8)

if matches.empty:
    st.warning("No matching address found.")
    st.stop()

choice = st.radio("Did you mean:", matches["Address"].tolist())

# Find the selected row
property_data = matches[matches["Address"] == choice].iloc[0]

latitude = property_data["Latitude"]
longitude = property_data["Longitude"]
neighborhood = property_data["nhood"]

# ------------------------------------------------------------------
# Property information
# ------------------------------------------------------------------

col1, col2 = st.columns(2)

with col1.container(border=True):

    st.subheader("Property record")

    details = {
        "Address": choice,
        "Neighborhood": neighborhood,
        "Latitude": latitude,
        "Longitude": longitude,}

    st.table(pd.DataFrame({"Value": details}))

# ------------------------------------------------------------------
# Rent estimate
# ------------------------------------------------------------------

def predict_rent():
    return random.randint(1000, 2000)


row2_col1, row2_col2 = st.columns(2)

with row2_col1.container(border=True):

    st.subheader("Suggested asking rent")

    predicted_rent = predict_rent()

    st.metric(
        "Recommended rent",
        f"${predicted_rent:,}/month"
    )

    st.caption(
        "This is a test prediction using the rent model."
    )


# ------------------------------------------------------------------
# Vacancy estimate
# ------------------------------------------------------------------

def predict_vacancy(price):
    return price ** 0.5


prices = list(range(predicted_rent - 500, predicted_rent + 501, 10))

vacancy_estimates = [predict_vacancy(price) for price in prices]

vacancy_data = pd.DataFrame({"Price": prices, "Vacancy": vacancy_estimates})


with row2_col2.container(border=True):

    st.subheader("Estimated vacancy by rent")

    st.line_chart(
        vacancy_data,
        x="Price",
        y="Vacancy"
    )

    st.caption(
        "Test vacancy estimates based on a square-root function."
    )


# ------------------------------------------------------------------
# Model explanation
# ------------------------------------------------------------------

with st.container(border=True):

    st.subheader("Why this number")

    st.write(
        "Once the rent model is connected, "
        "this section will explain which property "
        "characteristics are driving the estimate."
    )

Overwriting app.py


## Step 3: Launch
Run the cell below and wait for a link ending in `.trycloudflare.com`. Open it; there is no password page.

The app keeps running in the background after the cell finishes. Re-run this cell after editing `app.py`.

In [ ]:
!streamlit run app.py &>/content/logs.txt &
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
!cloudflared tunnel --url http://localhost:8501

(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.9.3) over (2026.9.3) ...
Setting up cloudflared (2026.9.3) ...
2026-09-30T20:58:22Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-30T20:58:22Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-30T20:58:26Z INF +-------------------------------------------------------------------------------